# Step 1: Reproduce AdaptVis (English, LLaVA-1.5-7B, VSR) with the original repo

**Kaggle settings:** Accelerator **GPU T4 x2** (one T4 is used) · Internet **on** ·
Add Input → Datasets → **`awsaf49/coco-2017-dataset`** (VSR images are COCO images).

The repo is run almost unchanged. Patches below:
1. load the model in **fp16** (the repo loads fp32 = 28 GB, which does not fit a 16 GB T4),
2. comment out `import clip` (only used for CLIP baselines; the `clip` package pinned on PyPI is not OpenAI's),
3. write attention dumps under `./outputs/attn` instead of the author's home folder.

Expected runtime on one T4: ~15–25 min for `base`, about the same for `scaling_vis`, ~2× that for `adapt_vis`.

In [ ]:
!git clone -q https://github.com/shiqichen17/AdaptVis.git /kaggle/working/AdaptVis
%cd /kaggle/working/AdaptVis
# Pinned to what the repo was written for. Kaggle's own torch is kept.
!pip install -q "transformers==4.39.1" "tokenizers==0.15.2" easydict
import transformers, torch, numpy as np
print(transformers.__version__, torch.__version__, np.__version__)
# If you see numpy-related import errors: !pip install -q "numpy<2"  then Run -> Restart and run all.

In [ ]:
import re, pathlib
p = pathlib.Path("model_zoo/llava15.py"); s = p.read_text()
s = s.replace("cache_dir=root_dir,ignore_mismatched_sizes=True).eval().to(device)",
              "cache_dir=root_dir,ignore_mismatched_sizes=True, torch_dtype=torch.float16).eval().to(device)")
s = s.replace('/home/user/shiqi/mmlm_mech/whatsup_vlms/outputs/', './outputs/attn/')
p.write_text(s)
assert s.count("torch_dtype=torch.float16") == 2, "fp16 patch did not apply - check model_zoo/llava15.py"

p = pathlib.Path("model_zoo/__init__.py"); s = p.read_text()
p.write_text(re.sub(r"^import clip", "# import clip", s, flags=re.M))
print("patched")

In [ ]:
# Data: VSR jsonl is downloaded by the repo; images are linked from the COCO Kaggle dataset.
import glob, os
coco = os.path.dirname(sorted(glob.glob("/kaggle/input/**/train2017", recursive=True))[0])
os.makedirs("data", exist_ok=True); os.makedirs("output", exist_ok=True); os.makedirs("outputs/attn", exist_ok=True)
for sub in ("train2017", "val2017"):
    if not os.path.exists(f"data/{sub}"):
        os.symlink(f"{coco}/{sub}", f"data/{sub}")
print(coco, os.listdir("data"))

Runs use `TEST_MODE=True`: the repo takes a random 20% of VSR as validation (indices saved to
`output/sampled_idx_VSR.npy`) and evaluates on the other 80%. Keep that file; notebook 02 reuses it.

VSR settings from `run.sh`: scaling_vis weight **0.5**. For adapt_vis, `run.sh` labels the last line
"Controlled_B" but it is the only remaining setting and appears to be VSR's: weight1 **0.5**, weight2 **1.2**, threshold **0.64**.
Check this against the paper's appendix before quoting numbers.

In [ ]:
%env TEST_MODE=True
!python main_aro.py --dataset=VSR --model-name=llava1.5 --download --method=base --num_workers=2 2>&1 | tail -8

In [ ]:
!python main_aro.py --dataset=VSR --model-name=llava1.5 --download --method=scaling_vis --weight=0.5 --num_workers=2 2>&1 | tail -8

In [ ]:
!python main_aro.py --dataset=VSR --model-name=llava1.5 --download --method=adapt_vis --weight1=0.5 --weight2=1.2 --threshold=0.64 --num_workers=2 2>&1 | tail -8

In [ ]:
# Collect results. Note: the repo's "precision"/"recall" fields are actually recall on true / recall on false items,
# so we recompute from the per-item generations.
import json, glob, pandas as pd
rows = []
for fp in sorted(glob.glob("outputs/results_VSR_*.json")):
    if fp.endswith("_scores.json"): continue
    r = pd.DataFrame(json.load(open(fp)))
    pred_yes = r.Generation.str.contains("Yes"); gold_yes = r.Golden.eq("Yes")
    rows.append({"run": os.path.basename(fp), "n": len(r), "acc": (pred_yes == gold_yes).mean(),
                 "acc_true_items": pred_yes[gold_yes].mean(), "acc_false_items": (~pred_yes[~gold_yes]).mean(),
                 "yes_rate": pred_yes.mean()})
summary = pd.DataFrame(rows); summary.to_csv("/kaggle/working/step1_adaptvis_repo_summary.csv", index=False)
summary

**Done when:** `adapt_vis` and `scaling_vis` beat `base` on accuracy, roughly in line with the paper's LLaVA-1.5 VSR table.
Save the notebook version (Save Version → *Save & Run All*) so `output/sampled_idx_VSR.npy` and the summary CSV are kept as outputs
for notebook 02.